## 1.Import Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

## 1.Paths

In [2]:
train_dir = r"D:\ProITBridge\CNN Scripting\train_data"
val_dir = r"D:\ProITBridge\CNN Scripting\val_data"
test_dir = r"D:\ProITBridge\CNN Scripting\test_data"

## 2.Parameters

In [3]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_CLASSES = 3

## 3.Load Training Data

In [4]:
train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True
)

Found 11 files belonging to 3 classes.


## 4.Load Validation Data

In [5]:

val_data = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

Found 5 files belonging to 3 classes.


## 5.Load Test Data

In [6]:
test_data = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

Found 4 files belonging to 3 classes.


## 6.Checking Class Names

In [7]:
class_names = train_data.class_names

print("Classes:", class_names)

Classes: ['ani', 'divo', 'sujin']


## 7.Normalize the images

In [8]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_data = train_data.map(
    lambda x, y: (normalization_layer(x), y)
)

val_data = val_data.map(
    lambda x, y: (normalization_layer(x), y)
)

test_data = test_data.map(
    lambda x, y: (normalization_layer(x), y)
)

## 8.Model Building

In [9]:
model = Sequential([

    # First convolution block
    Conv2D(
        32,
        (3, 3),
        activation="relu",
        input_shape=(128, 128, 3)
    ),
    MaxPooling2D((2, 2)),

    # Second convolution block
    Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),
    MaxPooling2D((2, 2)),

    # Third convolution block
    Conv2D(
        128,
        (3, 3),
        activation="relu"
    ),
    MaxPooling2D((2, 2)),

    # Convert feature maps to vector
    Flatten(),

    # Fully connected layer
    Dense(128, activation="relu"),

    # Prevent overfitting
    Dropout(0.5),

    # Output layer
    Dense(NUM_CLASSES, activation="softmax")
])

C:\Users\sujin\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## 9.Model Compiling

In [10]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       3,211,392 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             387 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,305,027 (12.61 MB)

 Trainable params: 3,305,027 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

## 10.Model training

In [16]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_accuracy",
    save_best_only=True
)

In [17]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 725ms/step - accuracy: 0.6364 - loss: 0.7732 - val_accuracy: 0.2000 - val_loss: 1.1862
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 417ms/step - accuracy: 1.0000 - loss: 0.5880 - val_accuracy: 0.0000e+00 - val_loss: 1.2994
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 514ms/step - accuracy: 1.0000 - loss: 0.3111 - val_accuracy: 0.0000e+00 - val_loss: 1.4946
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 422ms/step - accuracy: 0.9091 - loss: 0.3979 - val_accuracy: 0.0000e+00 - val_loss: 1.7029
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 377ms/step - accuracy: 0.8182 - loss: 0.4127 - val_accuracy: 0.2000 - val_loss: 2.1458
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 360ms/step - accuracy: 0.8182 - loss: 0.3424 - val_accuracy: 0.0000e+00 - val_loss: 2.5192
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 530ms/step - accuracy: 0.9091 - loss: 0.1423 - val_accuracy: 0.2000 - val_loss: 2.7644
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step - accuracy: 1.0000 - loss: 0.0307 - val_accuracy: 0.

## 11.Model Evaluation

In [18]:
test_loss, test_accuracy = model.evaluate(test_data)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.5000 - loss: 1.0286
Test Loss: 1.0285840034484863
Test Accuracy: 0.5


## 12.Saving the Model

In [19]:
model.save("people_classifier.keras")

## Predict a New Image

In [24]:
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array

img = load_img(
    r"D:\ProITBridge\CNN Scripting\test_data\ani\WhatsApp Image 2026-08-11 at 21.31.28.jpeg",
    target_size=IMG_SIZE
)

img_array = img_to_array(img)

img_array = img_array / 255.0

img_array = np.expand_dims(
    img_array,
    axis=0
)

prediction = model.predict(img_array)

predicted_class = np.argmax(prediction[0])

print("Predicted:", class_names[predicted_class])
print("Confidence:", prediction[0][predicted_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
Predicted: divo
Confidence: 0.40135372
